# Live Dashboard — Elliott Wave Trading Agent

Monitor paper/live trading: view open positions, recent signals, and P&L.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import matplotlib.pyplot as plt

from config.settings import get_settings, TradingMode
from data.yfinance_provider import YFinanceProvider
from signals.wave3_rsi import Wave3RSISignal
from execution.paper_broker import PaperBroker
from agents.orchestrator import Orchestrator
from agents.trade_planner import TradePlanner

plt.style.use('seaborn-v0_8-darkgrid')

## 1. Setup

In [ ]:
ASSETS = ['SPY', 'AAPL', 'NVDA']
TIMEFRAME = '1d'
CAPITAL = 10_000

settings = get_settings()
settings.trading_mode = TradingMode.PAPER

provider = YFinanceProvider()
signal = Wave3RSISignal()
broker = PaperBroker(initial_capital=CAPITAL)
orchestrator = Orchestrator(settings, provider, signal, broker)

## 2. Scan for Current Signals

In [ ]:
print('Scanning for signals...\n')
for symbol in ASSETS:
    df = provider.fetch_ohlcv(symbol, timeframe=TIMEFRAME)
    signals = signal.generate(df, symbol)
    
    if signals:
        latest = signals[-1]
        print(f'{symbol}: SIGNAL FOUND')
        print(f'  {latest}')
        print(f'  Risk/Reward: {latest.reward_risk_ratio:.2f}:1')
        
        # Generate trade plan
        planner = TradePlanner()
        plan = planner.create_plan(latest, CAPITAL)
        print(f'\n{plan.summary()}')
    else:
        print(f'{symbol}: No signals')
    print()

## 3. Run Single Trading Iteration

In [ ]:
# Run one iteration of the trading loop
orchestrator.run_trading_loop(
    assets=ASSETS,
    timeframe=TIMEFRAME,
    max_iterations=1,
)

## 4. Account Status

In [ ]:
account = broker.get_account()
print(f'Equity: ${account.equity:,.2f}')
print(f'Cash: ${account.cash:,.2f}')
print(f'Positions: {len(account.positions)}')

for pos in account.positions:
    print(f'\n  {pos.symbol}:')
    print(f'    Qty: {pos.quantity:.4f}')
    print(f'    Entry: ${pos.avg_entry_price:,.2f}')
    print(f'    Current: ${pos.current_price:,.2f}')
    print(f'    Unrealized P&L: ${pos.unrealized_pnl:,.2f}')